## Sample notebook from Angela

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import jmat
from scipy.linalg import dft
import torch

In [2]:
def pp_arr(arr, polar=True):
    from IPython.display import display, Latex

    if np.iscomplexobj(arr):
        if (abs(np.imag(arr))<1e-05).all():
            arr = np.real(arr)

    if len(arr.shape) == 1:
        arr = arr[None,:]

    if len(arr.shape) == 2:
        out = latex_matrix(arr, polar=polar)
        display(Latex(out))
    
    elif len(arr.shape) == 3:
        for i in range(arr.shape[0]):
            out = latex_matrix(arr[i,:,:], polar=polar)
            display(Latex(out))


def latex_matrix(mat, polar=True):
    out = '$$\\large \\begin{bmatrix} '

    for row in mat:
        if np.iscomplexobj(mat):
            if polar:
                out+=' & '.join([f'{np.abs(s):.2f}\\ e^{{{'+' if np.angle(s)>0 else '-'}i{np.abs(np.angle(s)):.2f}}}' if not np.isclose(s,0,atol=1e-05) else '0' for s in row])
            else:
                out+=' & '.join([f'{s:+.2f}' if not np.isclose(s,0,atol=1e-05) else '0' for s in row])
        elif np.issubdtype(mat.dtype, np.integer):
            out+=' & '.join([f'{s}' if not np.isclose(s,0,atol=1e-05) else '0' for s in row])
        else:
            out+=' & '.join([f'{s:.2f}' if not np.isclose(s,0,atol=1e-05) else '0' for s in row])

        out += ' \\\\'
    out += ' \\end{bmatrix} $$'

    return out


We construct a unitary for a d-level qudit with n pulses:

$$U=\prod_{i=0}^n e^{-iH(\vec{\phi_i})\theta_i}$$

where $H(\vec{\phi})=$
$$
\begin{pmatrix}
0 & \Omega_0 e^{-i\phi_0} & 0 & \dots \\
\Omega_0 e^{i\phi_0} & 0 & \Omega_1 e^{-i\phi_1} & \\
0 & \Omega_1 e^{i\phi_1} & 0 & \\
\vdots & & & \ddots
\end{pmatrix}
$$

and $\Omega_i=Jx_{ i,i+1}$

Distance between the found unitary V and ideal unitary U:

$D(V,U) = \sum_{i,j}|V_{i,j}-U_{i,j}|$



In [3]:
class Find_U(torch.nn.Module):
    def __init__(self, num_pulses, U):
        super().__init__()
        self.num_pulses = num_pulses

        self.d = U.shape[0]
        self.U = U
        self.jx = torch.tensor(jmat((self.d-1)/2, 'x').full(), dtype=torch.cfloat)
        
        self.phis = torch.nn.Parameter(torch.rand(num_pulses, self.d-1, dtype=torch.float))
        self.thetas = torch.nn.Parameter(torch.rand(num_pulses, dtype=torch.float))
        
    def forward(self):
        self.seq = torch.zeros(self.num_pulses, self.d, self.d, dtype=torch.cfloat)
        self.U_rf = torch.eye(self.d, dtype=torch.cfloat)
        for i in range(self.num_pulses):
            pulse_ham = self.jx * (torch.diag(torch.exp(-1j*self.phis[i,:]), diagonal=1) + torch.diag(torch.exp(1j*self.phis[i,:]), diagonal=-1))
            self.seq[i,:,:] = pulse_ham
            self.U_rf = torch.matrix_exp(-1j * pulse_ham * self.thetas[i]) @ self.U_rf

    def loss(self):
        return torch.sum((self.U_rf - self.U).abs())

def train(model, optimizer, epochs, progress=False):
    for epoch in range(epochs):
        optimizer.zero_grad()
        model.forward()
        loss = model.loss()
        loss.backward()
        optimizer.step()
        if progress:
            if epoch % 1000 == 0:
                print(f'Epoch {epoch}, Loss: {loss.item()}')
    return loss.item()

Example: Find a sequence that approximates the Quantum Fourier Transform for d=4

In [4]:
def qdft(d):
    mat = dft(d, scale='sqrtn')
    det = np.linalg.det(mat)
    return mat/(det**(1/d))

n = 4 # number of pulses
U = torch.tensor(qdft(n), dtype=torch.cfloat)

print(np.linalg.det(U))
model = Find_U(n, U)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

(0.9999999-2.0332722e-08j)


In [5]:
train(model, optimizer, 5000, progress=True)

Epoch 0, Loss: 9.942045211791992
Epoch 1000, Loss: 4.759828090667725
Epoch 2000, Loss: 1.0412229299545288
Epoch 3000, Loss: 0.3608110547065735
Epoch 4000, Loss: 0.0043651266023516655


0.002216821536421776

In [6]:
pp_arr(model.seq.clone().detach().numpy())

pp_arr(model.U_rf.clone().detach().numpy())

pp_arr(model.U.clone().detach().numpy())

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>